B22. Асимметрия конформности по направлению триажа.....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
PANEL_REL = {'P-base': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': PANELS_OTHER_DIR['P-weak'] + '/llm_as_judge/' + PANEL_FILE['P-weak'], 'P-mixed': PANELS_OTHER_DIR['P-mixed'] + '/llm_as_judge/' + PANEL_FILE['P-mixed'], 'P-strong': PANELS_OTHER_DIR['P-strong'] + '/llm_as_judge/' + PANEL_FILE['P-strong'], 'homog gpt': HOMOG_DIRS['gpt'] + '/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog qwen': HOMOG_DIRS['qwen'] + '/llm_as_judge/delphi_homog_qwen_noroles_judge.csv', 'homog gemini': HOMOG_DIRS['gemini'] + '/llm_as_judge/delphi_homog_gemini_noroles_judge.csv', 'homog deepseekv4': HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
def build(pname, rel):
    d = rd(rel)
    if d is None or 'Type' not in d.columns:
        return None
    ps = participants(d)
    truth = true_label(d).map(ORD)
    lab = {}
    for r in (1, 2, 3):
        for p in ps:
            c = 'r%d_emergency__%s' % (r, p)
            lab[r, p] = d[c].map(norm_label).map(ORD) if c in d.columns else pd.Series(index=d.index, dtype=float)
    R1 = pd.DataFrame({p: lab[1, p] for p in ps})
    med1 = R1.median(axis=1)
    maj1 = R1.mode(axis=1)
    maj1 = maj1[0] if len(maj1.columns) else pd.Series(index=d.index, dtype=float)
    cons = pred_triage(d).map(ORD)
    rr = pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce')
    rows = []
    for p in ps:
        l1 = lab[1, p]
        last = lab[3, p].where(lab[3, p].notna(), lab[2, p])
        last = last.where(last.notna(), l1)
        rows.append(pd.DataFrame({'panel': pname, 'case': d.index, 'pid': p, 'r1': l1.to_numpy(), 'last': last.to_numpy(), 'truth': truth.to_numpy(), 'med1': med1.to_numpy(), 'maj1': maj1.to_numpy(), 'consensus': cons.to_numpy(), 'rounds': rr.to_numpy()}))
    L = pd.concat(rows, ignore_index=True)
    return L.dropna(subset=['r1', 'last', 'truth'])
L = pd.concat([x for x in (build(p, r) for p, r in PANEL_REL.items()) if x is not None], ignore_index=True)
L['сдвиг'] = L['last'] - L['r1']
L['направление'] = np.select([L['сдвиг'] > 0, L['сдвиг'] < 0], ['вверх (срочнее)', 'вниз (спокойнее)'], 'без изменений')
print('наблюдений (кейс × участник):', len(L))
print('панелей:', L['panel'].nunique())


наблюдений (кейс × участник): 5415
панелей: 8


In [3]:
E = L[L['rounds'] > 1]
rows = []
for p, g in E.groupby('panel', observed=True):
    up = int((g['сдвиг'] > 0).sum())
    dn = int((g['сдвиг'] < 0).sum())
    tot = up + dn
    pv = float(stats.binomtest(up, tot, 0.5).pvalue) if tot else np.nan
    rows.append({'панель': p, 'наблюдений': len(g), 'сменили метку': tot, 'вверх': up, 'вниз': dn, 'доля вверх': up / tot if tot else np.nan, 'p (тест знаков)': pv})
S = pd.DataFrame(rows).set_index('панель')
show(S, 'H22.1 Направление смен мнения между раундами')
up, dn = (int((E['сдвиг'] > 0).sum()), int((E['сдвиг'] < 0).sum()))
print()
print('ИТОГО: вверх %d, вниз %d, доля вверх = %.3f' % (up, dn, up / (up + dn)))
print('точный биномиальный тест H0 «направления равновероятны»: p = %s' % fmt_p(stats.binomtest(up, up + dn, 0.5).pvalue))



H22.1 Направление смен мнения между раундами
------------------------------------------------------------
                  наблюдений  сменили метку  вверх  вниз  доля вверх  p (тест знаков)
панель                                                                               
P-base                   327             66     30    36      0.4545           0.5386
P-mixed                  261             44     12    32      0.2727           0.0037
P-strong                 303             62     24    38      0.3871           0.0980
P-weak                   326             48     16    32      0.3333           0.0293
homog deepseekv4         174             27     14    13      0.5185           1.0000
homog gemini             111             11      7     4      0.6364           0.5488
homog gpt                147             35     18    17      0.5143           1.0000
homog qwen                96             15      4    11      0.2667           0.1185

ИТОГО: вверх 125, вниз 183, доля

In [4]:
E2 = E.dropna(subset=['maj1']).copy()
E2['большинство относительно меня'] = np.select([E2['maj1'] > E2['r1'], E2['maj1'] < E2['r1']], ['большинство срочнее', 'большинство спокойнее'], 'совпадает')
sub = E2[E2['большинство относительно меня'] != 'совпадает'].copy()
sub['пошёл за большинством'] = np.where(E2.loc[sub.index, 'maj1'] > E2.loc[sub.index, 'r1'], sub['сдвиг'] > 0, sub['сдвиг'] < 0)
tab = pd.crosstab(sub['большинство относительно меня'], sub['пошёл за большинством'], normalize='index')
cnt = pd.crosstab(sub['большинство относительно меня'], sub['пошёл за большинством'])
T = pd.DataFrame({'наблюдений': cnt.sum(axis=1), 'пошли за большинством': cnt.get(True, 0), 'доля': tab.get(True, 0)})
show(T, 'H22.2 Подчинение большинству в зависимости от его направления')
if cnt.shape == (2, 2):
    orat = cnt.iloc[0, 1] * cnt.iloc[1, 0] / max(cnt.iloc[0, 0] * cnt.iloc[1, 1], 1e-09)
    print('\nотношение шансов (спокойное большинство / срочное): %.2f' % orat)
    print('точный тест Фишера: p = %s' % fmt_p(stats.fisher_exact(cnt.to_numpy())[1]))
    print()
    if orat > 1:
        print('ЗНАЧЕНИЕ БОЛЬШЕ 1 ЧИТАЕТСЯ ТАК: участник охотнее подчиняется')
        print('СПОКОЙНОМУ большинству, чем тревожному. То есть заразительнее')
        print('оказывается де-эскалация, а не перестраховка — направление,')
        print('ПРОТИВОПОЛОЖНОЕ исходной гипотезе H22.2.')
    else:
        print('Значение меньше 1: заразительнее «тревожное» большинство,')
        print('как и предполагала H22.2.')



H22.2 Подчинение большинству в зависимости от его направления
-------------------------------------------------------------
                               наблюдений  пошли за большинством   доля
большинство относительно меня                                          
большинство спокойнее                 143                    112 0.7832
большинство срочнее                   118                     80 0.6780

отношение шансов (спокойное большинство / срочное): 1.72
точный тест Фишера: p = 0.067

ЗНАЧЕНИЕ БОЛЬШЕ 1 ЧИТАЕТСЯ ТАК: участник охотнее подчиняется
СПОКОЙНОМУ большинству, чем тревожному. То есть заразительнее
оказывается де-эскалация, а не перестраховка — направление,
ПРОТИВОПОЛОЖНОЕ исходной гипотезе H22.2.


In [5]:
rows = []
for p, g in L.groupby('panel', observed=True):
    per = g.groupby('case').agg(med_r1=('r1', 'median'), med_last=('last', 'median'), cons=('consensus', 'first'), truth=('truth', 'first')).dropna()
    d1 = per['med_last'] - per['med_r1']
    d2 = per['cons'] - per['med_last']
    d3 = per['cons'] - per['med_r1']

    def sg(x):
        u, dn_ = (int((x > 0).sum()), int((x < 0).sum()))
        return (u, dn_, float(stats.binomtest(u, u + dn_, 0.5).pvalue) if u + dn_ else np.nan)
    u1, dn1, p1 = sg(d1)
    u2, dn2, p2 = sg(d2)
    u3, dn3, p3 = sg(d3)
    rows.append({'панель': p, 'кейсов': len(per), 'R1→последний: вверх/вниз': '%d/%d' % (u1, dn1), 'p1': p1, 'последний→консенсус: вверх/вниз': '%d/%d' % (u2, dn2), 'p2': p2, 'R1→консенсус: вверх/вниз': '%d/%d' % (u3, dn3), 'p3': p3, 'ср. сдвиг R1→консенсус': float(d3.mean())})
St = pd.DataFrame(rows).set_index('панель')
show(St, 'H22.3 На каком этапе появляется сдвиг вверх')
print()
print('Читается так: если значимая асимметрия появляется на этапе')
print('«последний→консенсус», сдвиг создаёт СИНТЕЗ (версия б).')
print('Если на этапе «R1→последний» — сдвиг создают РАУНДЫ (версия а).')
print()
print('ВАЖНОЕ УСТРОЙСТВО ДАННЫХ, обнаруженное здесь: колонка')
print('«последний→консенсус» почти везде равна 0/0. Это значит, что итоговая')
print('метка срочности ТОЖДЕСТВЕННА медиане голосов последнего раунда —')
print('то есть метка получается голосованием, а не пишется организатором.')
print('Следовательно канала «синтез escalates» для МЕТКИ не существует в')
print('принципе. Синтез влияет на текст протокола (и через него на рубрику,')
print('см. B11 и B21), но не на метку триажа.')



H22.3 На каком этапе появляется сдвиг вверх
------------------------------------------------------------
                  кейсов R1→последний: вверх/вниз     p1 последний→консенсус: вверх/вниз     p2 R1→консенсус: вверх/вниз     p3  ср. сдвиг R1→консенсус
панель                                                                                                                                                 
P-base               229                      9/5 0.4240                             0/0    NaN                      9/5 0.4240                  0.0175
P-mixed              229                      1/9 0.0215                             0/0    NaN                      1/9 0.0215                 -0.0349
P-strong             229                      2/7 0.1797                             0/0    NaN                      2/7 0.1797                 -0.0218
P-weak               229                      3/4 1.0000                             2/0 0.5000                      5/4 1.0000       

In [6]:
rows = []
for p, rel in PANEL_REL.items():
    d = rd(rel)
    t = true_label(d).map(ORD)
    cons = pred_triage(d).map(ORD)
    g = L[L['panel'] == p]
    med1 = g.groupby('case')['r1'].median()
    ix = t.dropna().index.intersection(cons.dropna().index).intersection(med1.dropna().index)
    t_, c_, m_ = (t.loc[ix], cons.loc[ix], med1.loc[ix])
    rows.append({'панель': p, 'кейсов': len(ix), 'недотриаж по медиане R1': float((m_ < t_).mean()), 'недотриаж финальный': float((c_ < t_).mean()), 'перетриаж по медиане R1': float((m_ > t_).mean()), 'перетриаж финальный': float((c_ > t_).mean()), 'Δ недотриаж': float((c_ < t_).mean() - (m_ < t_).mean()), 'Δ перетриаж': float((c_ > t_).mean() - (m_ > t_).mean())})
show(pd.DataFrame(rows).set_index('панель'), 'От медианы мнений R1 к финальному консенсусу')
print()
print('Отрицательная «Δ недотриаж» и положительная «Δ перетриаж» означают:')
print('протокол берёт мнения участников и сдвигает их в безопасную сторону.')
print('Это ровно тот эффект, который в B12 описан как консервативная')
print('операционная точка — здесь показано, что он возникает ВНУТРИ протокола.')



От медианы мнений R1 к финальному консенсусу
------------------------------------------------------------
                  кейсов  недотриаж по медиане R1  недотриаж финальный  перетриаж по медиане R1  перетриаж финальный  Δ недотриаж  Δ перетриаж
панель                                                                                                                                        
P-base               229                   0.0044               0.0131                   0.1747               0.2009       0.0087       0.0262
P-weak               229                   0.0044               0.0044                   0.1834               0.1878       0.0000       0.0044
P-mixed              229                   0.0087               0.0087                   0.1878               0.1528       0.0000      -0.0349
P-strong             229                   0.0131               0.0131                   0.1834               0.1616       0.0000      -0.0218
homog gpt            223           

In [7]:
RS = [1, 3, 5, 10, 20, 50]
rows = []
for p, rel in PANEL_REL.items():
    d = rd(rel)
    t = true_label(d).map(ORD)
    cons = pred_triage(d).map(ORD)
    g = L[L['panel'] == p]
    med1 = g.groupby('case')['r1'].median()
    ix = t.dropna().index.intersection(cons.dropna().index).intersection(med1.dropna().index)
    t_, c_, m_ = (t.loc[ix], cons.loc[ix], med1.loc[ix])
    r = {'панель': p}
    for R in RS:
        cost_med = R * float((m_ < t_).mean()) + float((m_ > t_).mean())
        cost_fin = R * float((c_ < t_).mean()) + float((c_ > t_).mean())
        r['R=%d' % R] = cost_fin - cost_med
    rows.append(r)
C = pd.DataFrame(rows).set_index('панель')
show(C, 'Изменение ожидаемых издержек от сдвига (отрицательное = сдвиг полезен)')
print()
print('Ищем R, при котором значения меняют знак: это порог, начиная с')
print('которого консервативный сдвиг протокола окупается.')



Изменение ожидаемых издержек от сдвига (отрицательное = сдвиг полезен)
----------------------------------------------------------------------
                     R=1     R=3     R=5    R=10    R=20    R=50
панель                                                          
P-base            0.0349  0.0524  0.0699  0.1135  0.2009  0.4629
P-weak            0.0044  0.0044  0.0044  0.0044  0.0044  0.0044
P-mixed          -0.0349 -0.0349 -0.0349 -0.0349 -0.0349 -0.0349
P-strong         -0.0218 -0.0218 -0.0218 -0.0218 -0.0218 -0.0218
homog gpt        -0.0090 -0.0090 -0.0090 -0.0090 -0.0090 -0.0090
homog qwen       -0.0131 -0.0131 -0.0131 -0.0131 -0.0131 -0.0131
homog gemini      0.0044  0.0044  0.0044  0.0044  0.0044  0.0044
homog deepseekv4  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000

Ищем R, при котором значения меняют знак: это порог, начиная с
которого консервативный сдвиг протокола окупается.


In [8]:
save(S.reset_index(), 'directional_triage_conformity_direction.csv')
save(St.reset_index(), 'directional_triage_conformity_stages.csv')
save(C.reset_index(), 'directional_triage_conformity_cost_shift.csv')
save(L, 'directional_triage_conformity_long.csv')


сохранено: out/directional_triage_conformity_direction.csv  (8 строк)
сохранено: out/directional_triage_conformity_stages.csv  (8 строк)
сохранено: out/directional_triage_conformity_cost_shift.csv  (8 строк)
сохранено: out/directional_triage_conformity_long.csv  (5415 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\directional_triage_conformity_long.csv'